In [ ]:
import sys
from pathlib import Path

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from pydantic import BaseModel, Field

# from faithfulmed.retrieval import retrieve_medical_context

In [ ]:
# source_metadata must use a typed model 
# using 'dict' will emit additionalProperties, which Gemini will reject with 400 error
class SourceReference(BaseModel):
    title: str
    url: str | None = None
    section: str | None = None
    distance: float | None = None

# output schema for Simplifer - pydantic
class SimplifiedOutput(BaseModel):
    """Structured output from the Simplifier agent."""

    simplified_text: str = Field(description="The 8th-grade level patient explanation.")
    source_metadata: list[SourceReference] = Field(
        description="The MedlinePlus passages the draft relied on."
    )
    glossary_terms_used: list[str] = Field(
        description="Medical terms explained from the glossary."
    )
    ungrounded_terms: list[str] = Field(
        description="Terms that had no glossary support and were paraphrased instead."
    )
    facts_covered: list[str] = Field(
        description="fact_id values the text covers. The Verifier checks this claim."
    )

# Prompt and completion tokens are not part of this object. The model cannot count them;
# the run record reads them from the runner's usage_metadata instead.

In [8]:
simplifier_agent = LlmAgent(
    name="Simplifier",
    model="gemini-3.5-flash-lite",
    instruction=(
        "You rewrite clinical facts as a plain-language patient explanation.\n"
        "Write at an 8th-grade reading level: short sentences, everyday words, active voice.\n\n"
        "The facts below are the only source of truth about this patient:\n"
        "{facts?}\n\n"
        "Call retrieve_medical_context for each medical term you need to explain. A glossary\n"
        "passage explains a word, never this patient: it must not add clinical detail, change\n"
        "a number, or drop a negation.\n\n"
        "When a term returns no passages, paraphrase it in the plainest words the facts\n"
        "support, invent no definition, and list the term in ungrounded_terms.\n\n"
        "Cover every fact whose must_keep is true, and report the ones you covered in\n"
        "facts_covered."
    ),
    tools=[FunctionTool(func=retrieve_medical_context)],
    output_schema=SimplifiedOutput,
    output_key="draft",
)

## Notes

**`SimplifiedOutput` is the output object, and `source_metadata` is a typed model inside it.** The Readability agent and the Verifier both read this object, so a field change will affect both agents downstream.

Token counts are not in the schema. The model cannot count tokens. Read them from the runner's `usage_metadata`.

**Not yet wired:** `{facts?}` fills from the session state key `facts`, which only the Extractor writes, so the placeholder renders empty until that pipeline runs.